# Estimasi Usia Wajah — Google Colab
UTKFace, MobileNetV3 Small, baseline, fine tuning, evaluasi, dan inferensi.

## Persiapan satu kali
1. Buat folder **My Drive / ComVis** di Google Drive.
2. Unggah `ComVis_Colab_Source.zip` dari repo ke folder tersebut.
3. Unggah arsip Kaggle UTKFace Aligned & Cropped ke folder yang sama;
   ubah **nama arsip** menjadi `dataset.zip` (jangan ubah nama gambar di dalamnya).
4. Di Colab pilih **Runtime → Change runtime type → GPU** (misalnya T4 jika tersedia).
5. Jalankan sel bootstrap di bawah, lalu sel proyek secara berurutan.

Dataset tidak perlu masuk GitHub. Arsip disalin dari Drive dan diekstrak ke disk
runtime; manifest, checkpoint, dan metrik disimpan di Drive. CPU juga dapat
digunakan, tetapi lebih lambat. GPU bergantung pada ketersediaan Colab.

Belum ada skor penelitian yang disertakan. Logbook tetap kosong.
Rujukan Colab: [FAQ resmi](https://research.google.com/colaboratory/faq.html).

## A. Sambungkan Drive dan tentukan arsip
Jalankan sel ini dan setujui sambungan Google Drive pada dialog Colab.
Sesuaikan nama folder/arsip jika kamu menggunakan lokasi lain.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/ComVis')
SOURCE_ZIP = DRIVE_PROJECT / 'ComVis_Colab_Source.zip'
DATASET_ZIP = DRIVE_PROJECT / 'dataset.zip'
for archive in (SOURCE_ZIP, DATASET_ZIP):
    if not archive.is_file():
        raise FileNotFoundError(f'Arsip belum tersedia: {archive}. Unggah ke folder ComVis di Drive.')

## B. Muat kode dan ekstrak dataset ke disk runtime
Arsip disalin dahulu ke `/content`, kemudian diekstrak di sana. Nama folder memakai
hash arsip supaya arsip berbeda tidak bercampur. Setelah runtime baru, jalankan
ulang sel A–B; tidak perlu mengunggah arsip lagi jika masih ada di Drive.

In [ ]:
"""Ekstraksi ZIP untuk bootstrap Colab dan dataset lokal."""
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath


def extract_zip(archive, destination, images_only=False):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as stream:
        members = []
        seen = set()
        for entry in stream.infolist():
            name = entry.filename
            relative = PurePosixPath(name)
            if (relative.is_absolute() or '..' in relative.parts or '\\' in name
                    or ':' in name or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError(f'Path ZIP tidak aman: {name}')
            target = (destination / name).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Path ZIP keluar folder tujuan: {name}')
            if entry.is_dir():
                continue
            if target in seen:
                raise ValueError(f'Path duplikat dalam ZIP: {name}')
            seen.add(target)
            if images_only and target.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                continue
            members.append((entry, target))
        if not members:
            raise ValueError('Arsip tidak berisi file yang diperlukan.')
        for entry, target in members:
            target.parent.mkdir(parents=True, exist_ok=True)
            with stream.open(entry) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
    return len(members)


import hashlib
import os
import sys

def archive_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def stage_archive(source, prefix, images_only=False):
    local_archive = Path('/content') / source.name
    shutil.copy2(source, local_archive)
    digest = archive_hash(local_archive)
    destination = Path('/content') / f'{prefix}_{digest[:12]}'
    marker = destination / '.archive_sha256'
    if not marker.is_file():
        count = extract_zip(local_archive, destination, images_only=images_only)
        marker.write_text(digest, encoding='utf-8')
        print(f'{prefix}: {count:,} file diekstrak.')
    elif marker.read_text(encoding='utf-8') != digest:
        raise RuntimeError('Hash folder berbeda. Gunakan runtime baru.')
    else:
        print(f'{prefix}: ekstraksi sebelumnya digunakan kembali.')
    return destination

ROOT = stage_archive(SOURCE_ZIP, 'ComVis_source')
DATA = stage_archive(DATASET_ZIP, 'UTKFace', images_only=True)
if not (ROOT / 'faceage' / 'training.py').is_file():
    raise RuntimeError('Paket kode tidak sesuai; gunakan ComVis_Colab_Source.zip yang disediakan.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
print('Kode:', ROOT)
print('Dataset:', DATA)

## C. Dependensi Colab dan pemeriksaan GPU
Sel ini menggunakan torch/torchvision dari runtime Colab sehingga tidak mengganti
instalasi GPU dengan paket CPU Windows. Jika import torchvision atau CUDA gagal,
pilih runtime GPU baru dan ulangi bootstrap.

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(ROOT / 'requirements-colab.txt')], check=True)
import torch
import torchvision
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU belum aktif; pilih Runtime → Change runtime type → GPU jika hendak memakai GPU.')

## 1. Lingkungan dan modul proyek
Gunakan folder repo sebagai working directory. Untuk lokasi lain, sesuaikan `ROOT`.
Bobot ImageNet diunduh ke cache lokal repo saat pertama diperlukan.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()  # diatur oleh bootstrap Colab
if not (ROOT / 'faceage').is_dir():
    raise RuntimeError('Buka notebook dari folder ComVis atau sesuaikan ROOT ke folder repo.')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
os.environ.setdefault('TORCH_HOME', str(ROOT / '.cache' / 'torch'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache' / 'matplotlib'))

import torch
from IPython.display import display, Image as DisplayImage
from faceage.data import prepare_data, load_manifest, manifest_fingerprint
from faceage.io import read_json, read_csv
from faceage.training import train, validate_config
from faceage.evaluation import compare_runs
from faceage.inference import Predictor, decode_image

print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
print('Repo:', ROOT)

## 2. Konfigurasi eksperimen
Split train/validation/test menargetkan 70/15/15 dengan seed 42; duplikat berada
dalam kelompok yang sama. Rasio aktual bergantung pada kelompok data.

Aktifkan `RUN_PREPARE` untuk audit awal, lalu `RUN_TRAINING` untuk menjalankan
sel B0, B1, M1, dan B1 control. Gunakan `RUN_PREFIX` baru untuk eksperimen baru;
folder hasil lama tidak ditimpa. `RUN_COMPARE` menghitung ulang evaluasi validasi.
`RUN_FINAL_TEST` hanya diaktifkan setelah keputusan model dan konfigurasi dibekukan.

Parameter default: RGB 224×224, AdamW, Smooth L1, maksimum 30 epoch per tahap,
batch 32, patience 5. Learning rate head 1e-4 dan backbone 1e-5.

**Untuk pelaksanaan pertama:** ubah `RUN_PREPARE`, `RUN_TRAINING`, dan
`RUN_COMPARE` menjadi `True`; `RUN_FINAL_TEST` tetap `False`. Manifest di Drive
dipakai kembali jika sudah ada. Jika mengganti dataset atau seed, gunakan folder
`PROCESSED` baru dan nama run baru agar seluruh model memakai split yang sama.
Hasil lengkap berada di `My Drive/ComVis/runs/`.


In [ ]:
# DATA ditentukan oleh bootstrap dari arsip Kaggle.
PROCESSED = DRIVE_PROJECT / 'processed'
MANIFEST = PROCESSED / 'manifest.csv'
RUN_PREFIX = 'colab01'
DEVICE = 'auto'  # auto / cpu / cuda
CONFIG = validate_config(read_json(ROOT / 'configs/default.json'))
RUNS = {method: DRIVE_PROJECT / 'runs' / f'{RUN_PREFIX}_{method}'
        for method in ('b0', 'b1', 'm1', 'b1_control')}
COMPARISON = DRIVE_PROJECT / 'runs' / f'{RUN_PREFIX}_comparison'

RUN_PREPARE = False
RUN_TRAINING = False
RUN_COMPARE = False
RUN_FINAL_TEST = False
RUN_SMOKE = False

print(json.dumps(CONFIG, indent=2))

# Sel training berikutnya dapat melewati tahap yang sudah selesai di Drive.
# Tahap terputus belum didukung resume epoch/optimizer: gunakan nama run baru.
def train_stage(method, init_checkpoint=None):
    if not RUN_TRAINING:
        print(f'{method.upper()}: RUN_TRAINING = False.')
        return None
    run = RUNS[method]
    metadata_path = run / 'run.json'
    if metadata_path.is_file():
        metadata = read_json(metadata_path)
        if metadata.get('status') != 'complete':
            raise RuntimeError(f'{run.name} terputus/belum selesai. Pilih RUN_PREFIX baru untuk mengulang.')
        if (metadata.get('method') != method or metadata.get('config') != CONFIG
                or metadata.get('manifest_fingerprint') != manifest_fingerprint(load_manifest(MANIFEST))):
            raise RuntimeError('Run tersimpan memakai konfigurasi/split berbeda. Pilih RUN_PREFIX baru.')
        checkpoint = run / ('b0.json' if method == 'b0' else 'best.pt')
        if not checkpoint.is_file():
            raise RuntimeError('Checkpoint run selesai hilang. Pilih RUN_PREFIX baru.')
        print(f'{method.upper()}: hasil selesai di Drive digunakan kembali.')
        return metadata
    return train(method, DATA, MANIFEST, run, CONFIG, DEVICE, init_checkpoint)


## 3. Audit data dan manifest
Audit memeriksa nama file/usia, gambar rusak, dimensi, duplikat piksel, dan kemiripan
dHash. Tinjau `invalid_files.csv` dan `near_duplicates.csv`. Kemiripan dHash tidak
menjamin identitas orang terpisah antar-split. Manifest tidak ditimpa otomatis.

In [ ]:
if RUN_PREPARE and not MANIFEST.is_file():
    audit = prepare_data(DATA, PROCESSED, seed=CONFIG['seed'], near_threshold=4)
    display(audit)
elif (PROCESSED / 'audit.json').is_file():
    display(read_json(PROCESSED / 'audit.json'))
else:
    print('Belum ada audit. Unduh dataset, lalu aktifkan RUN_PREPARE.')

rows = load_manifest(MANIFEST) if MANIFEST.is_file() else []
if rows:
    print('Jumlah citra:', len(rows))
    print('Fingerprint:', manifest_fingerprint(rows))
    print('Split:', {name: sum(row['split'] == name for row in rows)
                     for name in ('train', 'validation', 'test')})

## 4. Distribusi usia data
Periksa kelompok usia yang sedikit sebelum memilih eksperimen. Sel ini membaca
manifest tanpa menampilkan atau menyimpan ulang wajah dataset.

In [ ]:
if rows:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    for split in ('train', 'validation', 'test'):
        ages = [row['age'] for row in rows if row['split'] == split]
        ax.hist(ages, bins=list(range(0, 121, 5)), alpha=.45, label=split)
    ax.set(xlabel='Label usia (tahun)', ylabel='Jumlah citra', title='Distribusi label UTKFace')
    ax.legend()
    display(fig)
    plt.close(fig)
else:
    print('Grafik tersedia setelah manifest dibuat.')

## 5. Baseline B0 — median usia data latih
B0 tidak menggunakan citra. Median dihitung hanya dari data latih, dan dipakai
untuk mengetahui apakah model citra lebih baik daripada prediksi konstan.

In [ ]:
display(train_stage('b0'))

## 6. Baseline B1 — fitur pralatih dengan backbone beku
MobileNetV3 Small menggunakan ImageNet. Backbone beserta statistik BatchNorm
dibekukan; classifier/head regresi dilatih. Checkpoint terbaik dipilih berdasarkan
MAE validasi, bukan hasil test set.

In [ ]:
display(train_stage('b1'))

## 7. Metode M1 — fine tuning
M1 dimulai dari checkpoint B1 pada split dan konfigurasi yang sama. Tiga modul
terakhir backbone serta head dibuka. Sel ini membutuhkan B1 selesai terlebih dahulu.

In [ ]:
display(train_stage('m1', init_checkpoint=RUNS['b1'] / 'best.pt'))

## 8. Kontrol tambahan — B1 control
B1 control dimulai dari checkpoint B1 yang sama dengan M1, lalu meneruskan
pelatihan head dengan backbone tetap beku. Perbandingan ini membantu menilai
perubahan fine tuning setelah kedua model memperoleh tahap pelatihan tambahan.
Batas epoch sama; early stopping dapat membuat jumlah epoch aktual berbeda.

In [ ]:
display(train_stage('b1_control', init_checkpoint=RUNS['b1'] / 'best.pt'))

## 9. Evaluasi dan perbandingan validasi
Laporkan MAE, RMSE, median absolute error, MAE per kelompok usia, serta jumlah
data tiap kelompok. Target penurunan MAE 20% terhadap B0 adalah sasaran proposal,
bukan hasil yang diasumsikan. Semua model menggunakan manifest yang sama.

In [ ]:
if RUN_COMPARE:
    comparison = compare_runs(list(RUNS.values()), DATA, MANIFEST, COMPARISON,
                              split='validation', device=DEVICE)
    display(comparison)
else:
    print('Aktifkan RUN_COMPARE setelah seluruh model selesai dilatih.')

for method, run in RUNS.items():
    scores = run / 'evaluation_validation' / 'metrics.json'
    if scores.is_file():
        print(method.upper())
        display(read_json(scores))

## 10. Grafik dan analisis kasus
Grafik menggunakan prediksi mentah. Kasus gagal disimpan sebagai path dan galat,
bukan alasan kegagalan yang dikarang. Tinjau citra secara lokal, lalu isi analisis
manusia pada `docs/ERROR_ANALYSIS.md`.

In [ ]:
SELECTED_METHOD = 'm1'
evaluation_dir = RUNS[SELECTED_METHOD] / 'evaluation_validation'
for filename in ('learning_curve.png', 'prediction_vs_label.png',
                 'residual_histogram.png', 'age_group_mae.png'):
    path = evaluation_dir / filename
    if path.is_file():
        print(filename)
        display(DisplayImage(filename=str(path)))
worst = evaluation_dir / 'worst_examples.csv'
if worst.is_file():
    display(read_csv(worst)[:10])
else:
    print('Grafik dan kasus gagal belum tersedia untuk model terpilih.')

## 11. Evaluasi akhir test set
Setelah konfigurasi dipilih menggunakan validasi dan dibekukan, aktifkan
`RUN_FINAL_TEST`. Jangan memilih model atau menyetel hyperparameter berdasarkan
skor test. Evaluasi akhir seluruh metode dilakukan pada split test yang sama.

In [ ]:
if RUN_FINAL_TEST:
    display(compare_runs(list(RUNS.values()), DATA, MANIFEST, COMPARISON,
                         split='test', device=DEVICE, final_test=True))
else:
    print('Evaluasi test belum dijalankan; RUN_FINAL_TEST = False.')

## 12. Inferensi satu wajah baru
Isi `IMAGE_PATH` dengan foto JPG/PNG satu wajah yang sudah dipotong. Tidak ada
deteksi/crop otomatis. Gunakan foto dengan izin. Model penelitian terlatih diperlukan;
checkpoint sintetis ditolak untuk estimasi usia. Pembulatan/pembatasan tampilan
tidak digunakan saat menghitung metrik penelitian.

In [ ]:
IMAGE_PATH = None  # Contoh: ROOT / 'data/local/cropped_face.jpg'
CHECKPOINT = RUNS[SELECTED_METHOD] / 'best.pt'
if IMAGE_PATH is not None:
    image = decode_image(Path(IMAGE_PATH).read_bytes())
    display(image)
    predictor = Predictor(CHECKPOINT, device=DEVICE)
    display(predictor.predict(image))
else:
    print('Isi IMAGE_PATH setelah checkpoint model penelitian tersedia.')

## 13. Uji teknis opsional tanpa dataset wajah
Aktifkan `RUN_SMOKE` untuk menjalankan pipeline pada 30 gambar acak dan empat
metode satu epoch. Ini **bukan hasil estimasi usia wajah** atau bukti akurasi UTKFace.
Hasil disimpan di folder lokal yang diabaikan Git. Setiap pengulangan memakai
folder baru; tidak menimpa bukti lama.

In [ ]:
if RUN_SMOKE:
    import subprocess
    import uuid
    smoke_output = ROOT / 'artifacts/local' / f'notebook_smoke_{uuid.uuid4().hex[:8]}'
    subprocess.run([sys.executable, str(ROOT / 'scripts/smoke_test.py'),
                    '--output', str(smoke_output)], cwd=ROOT, check=True)
    display(read_json(smoke_output / 'smoke_summary.json'))
else:
    print('Smoke sintetis tidak dijalankan pada sesi ini.')

## 14. Inferensi dan penyimpanan hasil di Colab
Prediksi foto, grafik, dan evaluasi dilakukan langsung dalam notebook. Tidak perlu
menjalankan Streamlit atau membuat tunnel pada runtime Colab.

Manifest/audit ada di `My Drive/ComVis/processed/`. Model, history, metrik, dan grafik
ada di `My Drive/ComVis/runs/`. Unduh folder run bila ingin memakai model tersebut
pada aplikasi lokal kemudian. Simpan salinan notebook beserta output ke Drive.

Runtime Colab dapat berakhir; disk `/content` bersifat sementara. Checkpoint terbaik
disimpan saat validasi membaik. Tahap selesai dapat digunakan kembali setelah
bootstrap; tahap terputus belum mendukung resume epoch/optimizer dan perlu diulang
dengan nama run baru. Empat metode dapat dijalankan per sel, tidak harus sekali Run All.

Logbook di paket kode tetap kosong. Salin ke Drive/Sheets dan isi sendiri.
Daftar sumber dan panduan ada di `docs/REFERENCES.md` serta `docs/PANDUAN_UTS_UAS.md`.

Referensi Colab: [FAQ resmi](https://research.google.com/colaboratory/faq.html).


## 15. Unggah foto dan prediksi langsung di Colab
Setelah model penelitian tersedia, aktifkan `RUN_PHOTO_UPLOAD` lalu jalankan sel.
Pilih satu JPG/PNG wajah yang sudah dipotong. Foto diproses di memori dan disk
sementara Colab; sel ini tidak menyimpannya ke Drive. Prediksi dan gambar dapat ikut
tersimpan jika kamu menyimpan output notebook.

In [ ]:
RUN_PHOTO_UPLOAD = False
if RUN_PHOTO_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Pilih tepat satu foto wajah JPG/PNG.')
    name, content = next(iter(uploaded.items()))
    image = decode_image(content)
    display(image)
    predictor = Predictor(RUNS[SELECTED_METHOD] / 'best.pt', device=DEVICE)
    display(predictor.predict(image))
    # files.upload juga menulis file ke disk sementara Colab; tidak disalin ke Drive.
else:
    print('Aktifkan RUN_PHOTO_UPLOAD setelah checkpoint model tersedia.')